In [1]:
import pymc as pm
import numpy as np
import arviz as az
import pandas as pd
import numpy as np

rng = np.random.default_rng(6420)

# 6. Gesell*

Adapted from [Unit 9: Gesell.odc](https://raw.githubusercontent.com/areding/6420-pymc/main/original_examples/Codes4Unit9/Gesell.odc).

Gesell Developmental Schedules measure child development. The scoring is devised by the American child psychologist and pediatrician Arnold Gesell who founded the Clinic of Child Development at Yale in 1911 and directed it for many years. The Gesell Developmental Schedules are a gauge of the status of a child's motor and language development and personal-social and adaptive behaviors.

For 21 children the age in months at which they first spoke and their Gesell Adaptive Score, which is the result of an aptitude test taken much later was recorded. These data were originally collected by L. M. Linde of UCLA but were first published by M. R. Mickey, O. J. Dunn, and V. Clark, "Note on the use of stepwise regression in detecting outliers," *Computers and Biomedical Research*, 1 (1967), pp.105-111.

The data have been used by several authors, e.g., N. R. Draper and J. A. John, "Influential observations and outliers in regression,"
*Technometrics*, 23 (1981), pp. 21-26.

Let $X$ be the age in months a child speaks his/her first word and let $Y$ be the Gesell adaptive score, a measure of a child's aptitude (observed later on):

| Child  | 1  | 2  | 3  | 4  | 5  | 6  | 7  | 8  | 9  | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 | 20 | 21 |
|--------|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|
| Age    | 15 | 26 | 10 | 9  | 15 | 20 | 18 | 11 | 8  | 20 | 7  | 9  | 10 | 11 | 11 | 10 | 12 | 42 | 17 | 11 | 10 |
| Score  | 95 | 71 | 83 | 91 | 102| 87 | 93 |100 |104 | 94 |113 | 96 | 83 | 84 |102 |100 |105 | 57 |121 | 86 |100 |

A correlation of $r = -0.640$ indicates a  moderately strong negative relationship between age at first word spoken and Gesell score.

How does the child's aptitude change with how long it takes them to speak?

Are there any influential observations/potential outliers?

The solution follows a two stage modeling procedure.

1. Obtain mean ICPOs then take reciprocal of mean ICPOs (these are CPOs).  and consider them as an input for model B

2. Read in and calibrate CPOs with posterior ordinates.

In [2]:
data = pd.read_csv("../data/gesell.csv")
y = data["Score"].to_numpy()
x = data["Age"].to_numpy()

## Part 1

In [3]:
with pm.Model() as model_a:
    tau = pm.Exponential("tau", 0.001)
    s2 = pm.Deterministic("s2", 1 / tau)
    b = pm.Normal("b", 0, tau=0.00001, shape=2)

    m = b[0] + b[1] * x
    r = y - m
    f = pm.math.sqrt(tau / (2 * np.pi)) * pm.math.exp(-0.5 * tau * r**2)
    icpo = pm.Deterministic("icpo", 1 / f)
    pm.Normal("likelihood", mu=m, tau=tau, observed=y)

    trace = pm.sample(100000, target_accept=0.99, random_seed=rng)

NUTS[nutpie]: [tau, b]


Output()

In [4]:
icpo_df = az.summary(trace, var_names="icpo", round_to=5, filter_vars="like")
icpo_df

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
icpo[0],28.49149,4.76291,22.02737,36.83065,172785.38790,190150.20549,1.00001,0.01179,0.01111
icpo[1],48.04899,25.03746,27.51324,87.79041,170468.13679,215398.13099,1.00001,0.06021,0.23773
icpo[2],92.72087,51.51805,47.29908,176.77089,159838.13395,196511.60548,1.00004,0.11914,0.62089
icpo[3],40.56788,10.57210,28.38833,59.22507,146870.57244,184656.78812,1.00004,0.02807,0.04499
icpo[4],40.79738,8.67315,29.97727,56.07671,349986.48911,275843.16249,1.00001,0.01555,0.01968
icpo[5],28.34280,4.89251,21.77957,36.94929,162099.72358,179434.25289,1.00001,0.01255,0.01229
icpo[6],29.64986,5.14346,22.75283,38.67981,189259.72870,203690.80604,1.00001,0.01222,0.01204
icpo[7],28.93449,4.95254,22.26851,37.61913,164536.81688,183173.87850,1.00001,0.01265,0.01226
icpo[8],29.81861,5.57039,22.59704,39.58494,148356.94756,163701.12980,1.00002,0.01523,0.01738
icpo[9],34.90313,7.67596,25.53113,48.36578,243132.65573,240779.48063,1.00001,0.01630,0.02258


ICPO results match the original BUGS results from lecture.

In [5]:
cpo_means = 1 / icpo_df["mean"].to_numpy()
cpo_means

array([0.0350982 , 0.02081209, 0.01078506, 0.02465004, 0.02451138,
       0.03528233, 0.03372697, 0.03456083, 0.0335361 , 0.02865073,
       0.01923314, 0.03312782, 0.01078506, 0.01493539, 0.03237736,
       0.03515492, 0.02518928, 0.01504588, 0.00016066, 0.01911008,
       0.03515492])

## Part 2

In [6]:
mlcpo = -2 * np.log(cpo_means)
MPQ = np.sum(mlcpo)

In [7]:
with pm.Model() as model_b:
    tau = pm.Gamma("tau", 1, 0.002)
    s2 = pm.Deterministic("s2", 1 / tau)
    b = pm.Normal("b", 0, tau=0.00001, shape=2)

    m = b[0] + b[1] * x
    r = y - m
    f = pm.Deterministic(
        "f", (tau / (2 * np.pi)) ** 0.5 * pm.math.exp(-0.5 * tau * r**2)
    )
    c = pm.Deterministic("c", cpo_means / f)

    g1 = pm.Deterministic("g1", c * pm.math.log(c))
    g2 = -pm.math.log(c)
    g3 = 0.5 * np.absolute((c - 1))
    g4 = (c - 1) ** 2

    pm.Deterministic("s_g1", pm.math.sum(g1))
    pm.Deterministic("s_g2", pm.math.sum(g2))
    pm.Deterministic("s_g3", pm.math.sum(g3))
    pm.Deterministic("s_g4", pm.math.sum(g4))

    pm.Normal("likelihood", mu=m, tau=tau, observed=y)

    traceb = pm.sample(
        2000,
        target_accept=0.95,
        idata_kwargs=dict(log_likelihood=True),
        random_seed=rng,
    )

NUTS[nutpie]: [tau, b]


Output()

/Users/aaron/miniforge3/envs/pymc_f26/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1010: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_external_nuts(


In [8]:
az.summary(
    traceb, var_names="c", filter_vars="like", kind="stats_median", round_to=3
)

,median,mad,eti89_lb,eti89_ub
c[0],0.975,0.106,0.773,1.287
c[1],0.857,0.182,0.569,1.822
c[2],0.862,0.225,0.506,1.911
c[3],0.947,0.139,0.695,1.452
c[4],0.964,0.124,0.730,1.387
c[5],0.970,0.107,0.771,1.303
c[6],0.972,0.108,0.767,1.309
c[7],0.973,0.108,0.772,1.298
c[8],0.967,0.112,0.760,1.336
c[9],0.961,0.129,0.729,1.406


In [9]:
az.summary(traceb, var_names="s_g", filter_vars="like", kind="stats_median")

,median,mad,eti89_lb,eti89_ub
s_g1,-0.18,1.5,-2.3,10
s_g2,2.7,0.67,0.054,3.7
s_g3,2.2,0.61,1.4,5.5
s_g4,1.6,0.57,0.83,18


In [10]:
az.summary(traceb, var_names="g1", kind="stats_median")

,median,mad,eti89_lb,eti89_ub
g1[0],-0.024,0.1,-0.2,0.32
g1[1],-0.13,0.14,-0.32,1.1
g1[2],-0.13,0.17,-0.34,1.2
g1[3],-0.051,0.13,-0.25,0.54
g1[4],-0.035,0.12,-0.23,0.45
g1[5],-0.029,0.1,-0.2,0.34
g1[6],-0.028,0.1,-0.2,0.35
g1[7],-0.027,0.1,-0.2,0.34
g1[8],-0.032,0.11,-0.21,0.39
g1[9],-0.038,0.12,-0.23,0.48


In [11]:
az.loo(traceb, pointwise=True)

Computed from 8000 posterior samples and 21 observations log-likelihood matrix.

         Estimate       SE
elpd_loo   -82.88     5.15
p_loo        3.60        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)       21  100.0%
   (0.70, 1]   (bad)         0    0.0%
    (1, Inf)   (very bad)    0    0.0%

In [12]:
%load_ext watermark
%watermark -n -u -v -iv -p pytensor

Last updated: Tue, 10 Nov 2026

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.16.1

pytensor: 3.3.0

arviz : 1.3.0
numpy : 2.4.6
pandas: 3.0.5
pymc  : 6.3.2

